# 预测斜率、风险价格与中性化

轴与单位决定回归系数的含义。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

每期横截面回归后，再对斜率沿时间求平均。

In [ ]:
x=np.array([-1.,0.,1.]);returns=np.array([[-.01,0,.03],[-.02,0,.04]])
slopes=[]
for row in returns:
    xc=x-x.mean();yc=row-row.mean();slope=xc@yc/(xc@xc);slopes.append(slope)
    print('intercept, slope:',row.mean()-slope*x.mean(),slope)
print('time mean:',np.mean(slopes))

中性化为投影残差；并列秩要使用平均秩。

In [ ]:
signal=np.array([-2.,-1.,1.,2.]);size=np.array([-1.,-1.,1.,1.]);Z=np.column_stack([np.ones(4),size])
beta=np.linalg.lstsq(Z,signal,rcond=None)[0];residual=signal-Z@beta;future=np.array([-1.,1.,-2.,2.])
print('exposure, residual, orthogonality:',beta,residual,Z.T@residual)
print('Pearson, rank IC:',np.corrcoef(residual,future)[0,1],stats.spearmanr(residual,future).statistic)

经典两遍法先沿时间估计每个资产的 beta。

In [ ]:
factor=np.array([-.01,.01,.018]);exposure=np.array([.5,1,1.5]);alpha=.004
panel=alpha+factor[:,None]*exposure[None,:]
design=np.column_stack([np.ones(3),factor]);first=np.linalg.lstsq(design,panel,rcond=None)[0]
second=np.linalg.lstsq(np.column_stack([np.ones(3),first[1]]),panel.mean(axis=0),rcond=None)[0]
print('first-pass beta:',first[1],'second-pass intercept, price:',second)

## 自己试一试

将所有资产 alpha 加 0.01；风险价格与截距如何变化？

## 反馈

精确模型中截距增加 0.01，风险价格不变。若 alpha 在资产间变化并与 beta 相关，第二遍斜率会改变。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。